In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.6/386.6 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.9/231.9 kB 17.6 MB/s eta 0:00:00


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import  XGBClassifier
from sklearn.metrics import mean_squared_error
import re
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_regression
import optuna
import numpy as np
from sklearn.metrics import roc_auc_score, f1_score, precision_recall_curve, average_precision_score
from sklearn.utils import class_weight
import pandas as pd

In [ ]:
# Chargement des fichiers
X_train = pd.read_csv("/content/X_train_filtered_freq.csv")
X_test = pd.read_csv("/content/X_test_filtered_freq.csv")
Y_train = pd.read_csv("/content/Y_train_sinistre_2classes.csv")

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", Y_train.shape)

X_train shape: (383610, 71)
X_test shape: (95852, 71)
y_train shape: (383610, 7)


In [ ]:
from sklearn.metrics import roc_auc_score
# Extraction des cibles
y_train = Y_train["SINISTRE"]
y_train_full = Y_train  # pour accéder à NB_SINISTRES

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)

# ---------------------------------------------
# 🎯 Modèle Multiclasse Direct (0, 1+)
# ---------------------------------------------

# Espérance conditionnelle uniquement pour la classe 1+
mean_1plus = y_train_full.loc[y_train == 1, "NB_SINISTRES"].mean()

# Pondération selon formule demandée :
# E[NB_SINISTRES] = P(0) * 0 + P(1+) * E[Y | Y ≥ 1]
weights = np.array([0, mean_1plus])

# Split classique
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 300),
        "max_depth": trial.suggest_int("max_depth", 2, 6),
        "learning_rate": trial.suggest_float("lr", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample", 0.6, 1.0),
        "random_state": 42,
        "eval_metric": "auc"
    }

    model = XGBClassifier(objective="binary:logistic", **params)
    model.fit(X_tr, y_tr)

    proba = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, proba)

    return auc

# 🔍 Lancement Optuna
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)

print("🏆 Best AUC ROC:", study.best_value)
print("✅ Best params:", study.best_params)


X_train shape: (383610, 71)
X_test shape: (95852, 71)
y_train shape: (383610,)


[I 2025-04-17 11:43:49,383] A new study created in memory with name: no-name-dff19778-ede8-4a90-b1f2-3a8d2e55839b
[I 2025-04-17 11:44:08,459] Trial 0 finished with value: 0.8336618167111584 and parameters: {'n_estimators': 179, 'max_depth': 6, 'lr': 0.013008671117067851, 'subsample': 0.6145387873271264, 'colsample': 0.8807482239263533}. Best is trial 0 with value: 0.8336618167111584.
[I 2025-04-17 11:44:24,205] Trial 1 finished with value: 0.8324955023040872 and parameters: {'n_estimators': 230, 'max_depth': 4, 'lr': 0.021506031063244006, 'subsample': 0.8519480334577874, 'colsample': 0.6643849690625309}. Best is trial 0 with value: 0.8336618167111584.
[I 2025-04-17 11:44:33,315] Trial 2 finished with value: 0.8346916030919457 and parameters: {'n_estimators': 153, 'max_depth': 2, 'lr': 0.06478787428599167, 'subsample': 0.7864706811708155, 'colsample': 0.9734554065429808}. Best is trial 2 with value: 0.8346916030919457.
[I 2025-04-17 11:44:42,425] Trial 3 finished with value: 0.844015685

🏆 Best AUC ROC: 0.8440156859723381
✅ Best params: {'n_estimators': 174, 'max_depth': 3, 'lr': 0.16040903143259005, 'subsample': 0.7121763384741596, 'colsample': 0.7336635969776653}


In [ ]:
# 🔁 Réentraînement avec les meilleurs paramètres
final_model = XGBClassifier(objective="binary:logistic", **study.best_params)
final_model.fit(X_train, y_train)

# 📈 Prédictions sur X_test
proba_test = final_model.predict_proba(X_test)  # shape: (n_samples, 2)
weights = np.array([0, mean_1plus])

# 📊 Espérance du nb de sinistres
esp_nb_test = (proba_test * weights).sum(axis=1)
freq_pred = esp_nb_test / X_test["ANNEE_ASSURANCE"].values

# 🧾 Export final
results = pd.DataFrame({
    "ID": X_test["ID"].values,
    "ANNEE_ASSURANCE": X_test["ANNEE_ASSURANCE"].values,
    "FREQ_prediction": freq_pred
})

results.to_csv("/content/freq_2.csv", index=False)

/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [11:46:29] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "colsample", "lr" } are not used.

  warnings.warn(smsg, UserWarning)


KeyboardInterrupt: 

prise en compte de la disproportion des classes

In [ ]:
# ---------------------------------------------
# 🎯 Configuration initiale
# ---------------------------------------------
y_train = Y_train["SINISTRE"]  # 0 vs 1
y_train_full = Y_train  # Contient NB_SINISTRES

print("Shapes initiaux:")
print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("\nDistribution des classes (0 vs 1+):")
print(y_train.value_counts(normalize=True))

# Calcul de l'espérance conditionnelle pour 1+
mean_1plus = y_train_full.loc[y_train == 1, "NB_SINISTRES"].mean()
print(f"\nEspérance conditionnelle (1+): {mean_1plus:.2f} sinistres")

# ---------------------------------------------
# 🛠️ Préparation des données
# ---------------------------------------------
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train  # Préservation du ratio de classes
)

# Calcul des poids de classe (
# Poids
scale_pos_weight = len(y_tr[y_tr==0]) / len(y_tr[y_tr==1])  # Ratio 0/1+

# ---------------------------------------------
# 🎯 Fonction d'optimisation Optuna
# ---------------------------------------------
def objective(trial):
    params = {
        "objective": "binary:logistic",
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "learning_rate": trial.suggest_float("lr", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample", 0.6, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 1),
        "min_child_weight": trial.suggest_int("min_child", 1, 10),
        "scale_pos_weight": trial.suggest_float("pos_weight", scale_pos_weight*0.5, scale_pos_weight*1.5),
        "random_state": 42,
        "eval_metric": ["aucpr", "auc"]
    }

    model = XGBClassifier(**params)

    # Entraînemen
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    # Prédictions probabilistes
    proba = model.predict_proba(X_val)[:, 1]

    # Métriques principales
    auc = roc_auc_score(y_val, proba)
    ap = average_precision_score(y_val, proba)  # AUC-PR (meilleur pour déséquilibre)

    # On maximise une combinaison des deux métriques
    return 0.7 * ap + 0.3 * auc  # Poids plus fort sur AUC-PR

# 🔍 Lancement Optuna
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20, timeout=3600)

print("\n🔍 Résultats optimisation:")
print("Best score (combinaison AUC-PR + AUC):", study.best_value)
print("Meilleurs paramètres:", study.best_params)

Shapes initiaux:
X_train: (383610, 71) | y_train: (383610,)

Distribution des classes (0 vs 1+):
SINISTRE
0    0.994927
1    0.005073
Name: proportion, dtype: float64

Espérance conditionnelle (1+): 1.05 sinistres


[I 2025-04-17 11:46:44,436] A new study created in memory with name: no-name-5336648c-6256-4001-8e91-88b032b89a84
[I 2025-04-17 11:47:22,146] Trial 0 finished with value: 0.2698405582105723 and parameters: {'n_estimators': 186, 'max_depth': 6, 'lr': 0.020047301486134816, 'subsample': 0.8460215872076837, 'colsample': 0.6867212996293179, 'gamma': 0.022720904252791274, 'min_child': 6, 'pos_weight': 167.3348998476378}. Best is trial 0 with value: 0.2698405582105723.
[I 2025-04-17 11:49:03,012] Trial 1 finished with value: 0.26051790858189533 and parameters: {'n_estimators': 465, 'max_depth': 6, 'lr': 0.04262655019797433, 'subsample': 0.831688467802184, 'colsample': 0.769690000773206, 'gamma': 0.6023374337963727, 'min_child': 5, 'pos_weight': 131.39711542966515}. Best is trial 0 with value: 0.2698405582105723.
[I 2025-04-17 11:49:34,464] Trial 2 finished with value: 0.26951219567198176 and parameters: {'n_estimators': 153, 'max_depth': 7, 'lr': 0.03307281900930768, 'subsample': 0.8582945265


🔍 Résultats optimisation:
Best score (combinaison AUC-PR + AUC): 0.27020240240609733
Meilleurs paramètres: {'n_estimators': 176, 'max_depth': 4, 'lr': 0.04175928696671633, 'subsample': 0.9768236773771529, 'colsample': 0.7040615833152815, 'gamma': 0.17252102815097609, 'min_child': 7, 'pos_weight': 143.19802582308336}


In [ ]:
# ---------------------------------------------
# 🚀 Entraînement final avec meilleurs paramètres
# ---------------------------------------------
best_params = study.best_params.copy()
best_params.update({
    "objective": "binary:logistic",
    "random_state": 42,
    "eval_metric": ["aucpr", "auc"]
})

final_model = XGBClassifier(**best_params)
final_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=True
)

# ---------------------------------------------
# 📊 Évaluation complète
# ---------------------------------------------
from sklearn.metrics import classification_report, confusion_matrix

y_pred_proba = final_model.predict_proba(X_val)[:, 1]

# Optimisation du seuil pour F1-score
precisions, recalls, thresholds = precision_recall_curve(y_val, y_pred_proba)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
best_threshold = thresholds[np.argmax(f1_scores)]
y_pred = (y_pred_proba >= best_threshold).astype(int)

print("\n📊 Performance finale:")
print(classification_report(y_val, y_pred, target_names=["0 sinistre", "1+ sinistres"]))
print("\nMatrice de confusion:")
print(confusion_matrix(y_val, y_pred))

# ---------------------------------------------
# 💡 Application actuarielle
# ---------------------------------------------
def predict_expected_claims(model, X, threshold=best_threshold):
    """Prédit l'espérance de sinistres en combinant classification et espérance conditionnelle."""
    proba_1plus = model.predict_proba(X)[:, 1]
    predictions_1plus = (proba_1plus >= threshold).astype(int)
    return predictions_1plus * mean_1plus  # E[N] = P(1+) * E[N|1+]


expected_claims = predict_expected_claims(final_model, X_val)
print("\nEspérance prédite pour les 5 premiers cas (val):")
print(expected_claims[:5])
print("Valeurs réelles (y_val):")
print(y_val.head().values)

[0]	validation_0-aucpr:0.01685	validation_0-auc:0.79866


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [12:04:42] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "colsample", "lr", "min_child", "pos_weight" } are not used.

  warnings.warn(smsg, UserWarning)


[1]	validation_0-aucpr:0.01883	validation_0-auc:0.80864
[2]	validation_0-aucpr:0.01920	validation_0-auc:0.81373
[3]	validation_0-aucpr:0.01951	validation_0-auc:0.81555
[4]	validation_0-aucpr:0.01984	validation_0-auc:0.81681
[5]	validation_0-aucpr:0.01996	validation_0-auc:0.82134
[6]	validation_0-aucpr:0.02027	validation_0-auc:0.82434
[7]	validation_0-aucpr:0.02035	validation_0-auc:0.82700
[8]	validation_0-aucpr:0.02195	validation_0-auc:0.82766
[9]	validation_0-aucpr:0.02291	validation_0-auc:0.83067
[10]	validation_0-aucpr:0.02333	validation_0-auc:0.83103
[11]	validation_0-aucpr:0.02432	validation_0-auc:0.83750
[12]	validation_0-aucpr:0.02451	validation_0-auc:0.83838
[13]	validation_0-aucpr:0.02564	validation_0-auc:0.83865
[14]	validation_0-aucpr:0.02561	validation_0-auc:0.84038
[15]	validation_0-aucpr:0.02732	validation_0-auc:0.84160
[16]	validation_0-aucpr:0.02736	validation_0-auc:0.84326
[17]	validation_0-aucpr:0.02787	validation_0-auc:0.84486
[18]	validation_0-aucpr:0.02879	validati

In [ ]:
# ---------------------------------------------
# 💾 Injection des prédictions + Sauvegarde
# ---------------------------------------------

# 🎯 On injecte les prédictions dans un dataframe
results = X_test[["ID", "ANNEE_ASSURANCE"]].copy()

# Prédiction finale sur l'ensemble test filtré
X_test_filtered = X_test.loc[X_test.index]
pred_freq_test = predict_expected_claims(final_model, X_test_filtered)

# Injection dans les lignes concernées
results.loc[X_test_filtered.index, "FREQ_prediction"] = pred_freq_test

results.to_csv("/content/freq.csv", index=False)
print("\n✅ Fichier 'freq.csv' sauvegardé avec les prédictions d'espérance de fréquence.")



✅ Fichier 'freq.csv' sauvegardé avec les prédictions d'espérance de fréquence.
